# [개념 정리]
### 7.5 부스팅
* 부스팅: 약한 학습기를 여러 개 연결하여 강한 학습기를 만드는 앙상블 방법을 말함
* 부스팅 방법: 에이다부스트와 그레디언트 부스팅

  ### 7.5.1 에이다부스트
  * 이전 모델이 과소접합했던 훈련 샘플의 가중치는 높이는 것으로 새로운 예측기는 학습하기 어려운 샘플에 점점 더 맞추는 방식임.

  ### 7.5.2 그레디언트 부스팅
  * 그레디언트 부스팅: 앙상블에 이전까지 오차를 보정하도록 예측기를 순차적으로 추가하며 예측기가 만든 잔여 오차에 새로운 학습기를 학습시킴
  * learning_Rate 매개변수: 각 트리의 기여 정도를 조정
  * subsample 매개변수: 각 트리가 훈련할 때 사용할 훈련 샘플의 비율을 지정

### 7.6 스태킹
* 앙상블에 속한 모든 예측기의 예측을 취합하는 간단한 함수를 사용하는 대신 취합하는 모델을 훈련시킬 수 없을까라는 기본 아이디어로 출발함.

---
#[코드 필사]


In [1]:
import warnings
warnings.filterwarnings('ignore')

# import package
import numpy as np
import os

# 5장에서의 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
X,y = make_moons(n_samples=100, noise=0.15)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [2]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200,
    algorithm="SAMME", learning_rate=0.5)

ada_clf.fit(X_train, y_train)

AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=200)

In [3]:
from sklearn.tree import DecisionTreeRegressor
tree_reg1=DecisionTreeRegressor(max_depth=2)
tree_reg1.fit(X,y)


DecisionTreeRegressor(max_depth=2)

In [4]:
y2=y-tree_reg1.predict(X)
tree_reg2=DecisionTreeRegressor(max_depth=2)
tree_reg2.fit(X,y2)

DecisionTreeRegressor(max_depth=2)

In [5]:
y3=y2-tree_reg1.predict(X)
tree_reg3=DecisionTreeRegressor(max_depth=2)
tree_reg3.fit(X,y3)

DecisionTreeRegressor(max_depth=2)

In [6]:
# y_pred=sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))

In [7]:
from sklearn.ensemble import GradientBoostingRegressor

gbrt=GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0)
gbrt.fit(X,y)

GradientBoostingRegressor(learning_rate=1.0, max_depth=2, n_estimators=3)

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val=train_test_split(X,y)

gbrt=GradientBoostingRegressor(max_depth=2, n_estimators=120)
gbrt.fit(X_train, y_train)

errors=[mean_squared_error(y_val, y_pred)
        for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators=np.argmin(errors)+1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators)
gbrt_best.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=2, n_estimators=np.int64(111))

In [9]:
gbrt=GradientBoostingRegressor(max_depth=2, warm_start=True)

min_val_error=float("inf")
error_going_up=0
for n_estimators in range(1,120):
  gbrt.n_estimators=n_estimators
  gbrt.fit(X_train, y_train)
  y_pred=gbrt.predict(X_val)
  val_error=mean_squared_error(y_val, y_pred)
  if val_error> min_val_error:
    min_val_error=val_error
    error_going_up=0
  else:
    error_going_up +=1
    if error_going_up ==5:
      break # 조기 종료

In [10]:
import xgboost
xgb_reg=xgboost.XGBRegressor()
xgb_reg.fit(X_train, y_train)
y_pred=xgb_reg.predict(X_val)

In [12]:
xgb_reg = xgboost.XGBRegressor(early_stopping_rounds=2)

xgb_reg.fit(X_train, y_train,
            eval_set = [(X_val, y_val)])
y_pred = xgb_reg.predict(X_val)

[0]	validation_0-rmse:0.43916
[1]	validation_0-rmse:0.39502
[2]	validation_0-rmse:0.37803
[3]	validation_0-rmse:0.37445
[4]	validation_0-rmse:0.37117
[5]	validation_0-rmse:0.36990
[6]	validation_0-rmse:0.37215
[7]	validation_0-rmse:0.36915
[8]	validation_0-rmse:0.37083
